# VeloceAI — temporary Colab engine
This version avoids BasicSR. Run the three code cells in order. Keep this Colab session open while using the site.
For faster video enhancement, select Runtime → Change runtime type → T4 GPU (or another available GPU) before running the cells. CPU mode is supported but can be much slower, especially for video.

In [ ]:
!apt-get -qq update && apt-get -qq install -y ffmpeg
!pip -q install fastapi uvicorn python-multipart pillow opencv-python-headless spandrel
import urllib.request, pathlib, os, subprocess, sys
MODEL='/content/RealESRGAN_x4plus.pth'
if not pathlib.Path(MODEL).exists():
    urllib.request.urlretrieve('https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth', MODEL)
print('Dependencies and model downloaded:', pathlib.Path(MODEL).stat().st_size, 'bytes')
import torch
print('PyTorch:',torch.__version__,'GPU:',torch.cuda.is_available())
from spandrel import ModelLoader
m=ModelLoader().load_from_file(MODEL)
print('Model loaded:',m.__class__.__name__)


In [ ]:
import urllib.request, subprocess, sys, time, pathlib
# Stop the previous server process launched by this cell before rebinding port 8000.
try:
    if server.poll() is None:
        server.terminate()
        server.wait(timeout=8)
except (NameError, Exception):
    pass
urllib.request.urlretrieve('https://raw.githubusercontent.com/rezeyuri-dot/veloceai/main/colab_server.py','/content/colab_server.py')
server_log=open('/content/veloceai-server.log','w')
server=subprocess.Popen([sys.executable,'/content/colab_server.py'],stdout=server_log,stderr=subprocess.STDOUT)
for i in range(45):
    try:
        import json
        health=json.loads(urllib.request.urlopen('http://127.0.0.1:8000/health',timeout=3).read())
        print('Health:',health)
        if health.get('ok'): break
    except Exception: pass
    time.sleep(2)
else:
    print(pathlib.Path('/content/veloceai-server.log').read_text(errors='ignore')[-5000:])
    raise RuntimeError('Server failed to start')


In [ ]:
import os, subprocess, time, re, pathlib, urllib.request
cloudflared='/usr/local/bin/cloudflared'
if not pathlib.Path(cloudflared).exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',cloudflared)
    os.chmod(cloudflared,0o755)
tunnel_log=open('/content/veloceai-tunnel.log','w')
tunnel=subprocess.Popen([cloudflared,'tunnel','--url','http://127.0.0.1:8000','--no-autoupdate'],stdout=tunnel_log,stderr=subprocess.STDOUT)
public_url=None
for i in range(60):
    time.sleep(2)
    text=pathlib.Path('/content/veloceai-tunnel.log').read_text(errors='ignore')
    match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',text)
    if match: public_url=match.group(0); break
if not public_url: raise RuntimeError('Tunnel URL not found; inspect /content/veloceai-tunnel.log')
print('VeloceAI temporary API URL:',public_url)
print('Health check:',urllib.request.urlopen(public_url+'/health',timeout=20).read().decode())
